In [5]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

*Scrapping et nettoyage des données des divers urls*

In [6]:
urls_df = {
        'url_1': 'https://sn.coinafrique.com/categorie/chiens',
        'df_1' :None,
        'url_2': 'https://sn.coinafrique.com/categorie/moutons',
        'df_2' :None,
        'url_3': 'https://sn.coinafrique.com/categorie/autres-animaux',
        'df_3' :None,
        'url_4': 'https://sn.coinafrique.com/categorie/poules-lapins-et-pigeons',
        'df_4' :None
    }

for i in range(1, 5):
    data = []
    response =  requests.get(urls_df[f'url_{i}'])
    bsp = bs(response.text, 'html.parser')
    containers = bsp.find_all('div', class_='col s6 m4 l3')
    for item in containers:
        try:
            image_url = item.find('img', class_='ad__card-img').attrs['src']
            price = item.find('p', class_='ad__card-price').text.strip()
            adresse = item.find('p', class_='ad__card-location').find('span').text
            info = {
                "Image_lien": image_url,
                "Prix": price,
                "Adresse": adresse,
            }
            if i == 4 :
                
                details_page_url = item.find('a', class_='card-image ad__card-image waves-block waves-light')['href']
                details_page_response = requests.get(f'https://sn.coinafrique.com{details_page_url}')
                details_page_bsp = bs(details_page_response.text, 'html.parser')
                
                element_detail = details_page_bsp.find('div', class_='ad__info__box ad__info__box-descriptions')
                if element_detail:
                    paragraphs = element_detail.find_all('p')
                    if len(paragraphs) >= 2:
                        info['Détail'] = paragraphs[1].text.strip().replace('\r\n' , ' ').replace('\r\r' , ' ')
                else:
                    info['Détail'] = ""
            else:
                name = item.find('p', class_='ad__card-description').text.strip()
                info['Nom'] = name
            
            data.append(info)
        except Exception as e:
            raise(e)
   
    urls_df[f'df_{i}'] = pd.DataFrame(data)

*Génération des fichiers cvs à partir des dataFrames*

In [7]:

for i in range(1, 5):
    match i:
        case 1:
            urls_df[f'df_{i}'].to_csv('cleaned_data/chiens.csv', index=False)
        case 2:
            urls_df[f'df_{i}'].to_csv('cleaned_data/moutons.csv',index=False)
        case 3:
            urls_df[f'df_{i}'].to_csv('cleaned_data/autres_animaux.csv',index=False)
        case 4:
            urls_df[f'df_{i}'].to_csv('cleaned_data/poules_lapins_et_pigeons.csv',index=False)
            